In [ ]:
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T

In [ ]:
session.use_database("AIRBNB_INVESTMENT_DB")
session.use_schema("BRONZE")

In [ ]:
table_list = (
    session.sql("""
        SELECT TABLE_NAME 
        FROM AIRBNB_INVESTMENT_DB.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'BRONZE'
        AND TABLE_NAME LIKE '%_REVIEWS'
    """).collect()
)

cities = [row["TABLE_NAME"].replace("_REVIEWS", "").lower() for row in table_list]
print(f"Found {len(cities)} cities: {cities}")

In [ ]:
for city in cities:
    df = session.table(f"Bronze.{city} reviews")

In [ ]:
expected_cols = {"listing_id", "id", "date", "reviewer_id", "reviewer_name", "comments"}
actual_cols   = set(df.columns)
missing_cols  = expected_cols - actual_cols

if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")
else:
    print(f"Schema check passed — all expected columns present")

df = (
    df
    .withColumn("listing_id",     F.col("listing_id").cast(T.LongType()))
    .withColumn("id",             F.col("id").cast(T.LongType()))
    .withColumn("reviewer_id",    F.col("reviewer_id").cast(T.LongType()))
    .withColumn("date",           F.to_date(F.col("date")))        # handles 'YYYY-MM-DD' strings
    .withColumn("reviewer_name",  F.col("reviewer_name").cast(T.StringType()))
    .withColumn("comments",       F.col("comments").cast(T.StringType()))
)

In [ ]:
critical_cols     = ["listing_id", "id", "date", "reviewer_id"]
non_critical_cols = ["reviewer_name", "comments"]

print("\nNull counts per column")
for col in critical_cols + non_critical_cols:
    null_count = df.filter(F.col(col).isNull()).count()
    print(f"{col}: {null_count} nulls")

before_null_drop = df.count()
df = df.filter(
    F.col("listing_id").isNotNull()  &
    F.col("id").isNotNull()          &
    F.col("date").isNotNull()        &
    F.col("reviewer_id").isNotNull()
)
after_null_drop = df.count()
print(f"\nDropped {before_null_drop - after_null_drop} rows with nulls in critical columns")


In [ ]:
total_rows    = df.count()
distinct_rows = df.dropDuplicates(["id"]).count()
dupe_count    = total_rows - distinct_rows

if dupe_count > 0:
    print(f"Found {dupe_count} duplicate review IDs — keeping first occurrence")
    from snowflake.snowpark.window import Window
    dedup_window = Window.partitionBy("id").orderBy("date")
    df = (
        df
        .withColumn("_row_num", F.row_number().over(dedup_window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )
else:
    print(f"No duplicate review IDs found")

combo_dupes = (
    total_rows
    - df.dropDuplicates(["listing_id", "reviewer_id", "date"]).count()
)
if combo_dupes > 0:
    print(f"{combo_dupes} rows share the same (listing_id, reviewer_id, date) — may indicate same reviewer reviewing twice in one day")


In [ ]:
future_date   = F.lit("2025-12-31").cast(T.DateType())
airbnb_launch = F.lit("2008-01-01").cast(T.DateType())

early_dates  = df.filter(F.col("date") < airbnb_launch).count()
future_dates = df.filter(F.col("date") > future_date).count()

if early_dates > 0:
    print(f"{early_dates} reviews dated before Airbnb launched (pre-2008)")
if future_dates > 0:
    print(f"{future_dates} reviews with future dates")

df = df.filter(
        (F.col("date") >= airbnb_launch) &
        (F.col("date") <= future_date)
    )


In [ ]:
df = (
    df
    .withColumn("reviewer_name", F.trim(F.col("reviewer_name")))
    .withColumn("comments",      F.trim(F.col("comments")))
)

df = (
    df
    .withColumn("reviewer_name",
        F.when(F.col("reviewer_name") == "", None)
         .when(F.col("reviewer_name").rlike("^\\s+$"), None)
         .otherwise(F.col("reviewer_name")))
    .withColumn("comments",
        F.when(F.col("comments") == "", None)
         .when(F.col("comments").rlike("^\\s+$"), None)
         .otherwise(F.col("comments")))
)

before_short = df.count()
    df = df.filter(
        F.col("comments").isNull() |
        (F.length(F.col("comments")) >= 5)
    )
    short_dropped = before_short - df.count()
    if short_dropped > 0:
        print(f"Dropped {short_dropped} rows with comments under 5 characters")


In [ ]:
before_id = df.count()
    df = df.filter(
        (F.col("listing_id")  > 0) &
        (F.col("reviewer_id") > 0) &
        (F.col("id")          > 0)
    )
    id_dropped = before_id - df.count()
    if id_dropped > 0:
        print(f"Dropped {id_dropped} rows with invalid IDs <= 0")

In [ ]:
review_volume = df.groupBy("listing_id").agg(
        F.count("id").alias("total_reviews"),
        F.countDistinct("reviewer_id").alias("unique_reviewers"),
        F.min("date").alias("first_review_date"),
        F.max("date").alias("last_review_date"),
    )

    comments_grouped = (
        df
        .filter(F.col("comments").isNotNull())
        .groupBy("listing_id")
        .agg(
            F.array_agg(F.col("comments")).alias("comments_list"),
            F.count("comments").alias("comment_count"),
        )
    )

In [ ]:
silver = review_volume.join(comments_grouped, on="listing_id", how="left")

    silver = silver.withColumn(
        "date_range",
        F.concat(
            F.date_format(F.col("first_review_date"), "dd/MM/yyyy"),
            F.lit(" - "),
            F.date_format(F.col("last_review_date"),  "dd/MM/yyyy"),
        )
    )

    silver = silver.withColumn(
        "comment_coverage_flag",
        F.when(F.col("comment_count").isNull() | (F.col("comment_count") == 0),
               F.lit("NO_COMMENTS"))
         .otherwise(F.lit("OK"))
    )

    silver = silver.select(
        "listing_id",
        "total_reviews",
        "unique_reviewers",
        "first_review_date",
        "last_review_date",
        "date_range",
        "comment_count",
        "comments_list",
        "comment_coverage_flag",
    

In [ ]:
session.use_schema("SILVER")
output_table = f"{city.upper()}_REVIEWS"
silver.write.mode("overwrite").save_as_table(output_table)
print(f"Saved to AIRBNB_INVESTMENT_DB.SILVER.{output_table}")